# CMTFusion fine-tuned on the Blender renders (v1)

Fine-tunes CMTFusion on the **independent snapshots** of `batch_3` and `batch_4`, then fuses every **scene** of
`batch_1` and `batch_2`, which the model never sees in training. `cmtfusion.ipynb` (published weights, no training) is
the baseline and is not changed by this notebook.

- **Method:** CMTFusion (Park, Vien & Lee, IEEE TCSVT 2023) from [seonghyun0108/CMTFusion](https://github.com/seonghyun0108/CMTFusion).
  Code is cloned on first run into `fusion/methods/_vendor/CMTFusion/`; the released weights (`pretrained.zip`, 2.3 MB,
  Google Drive link in the repo README) are downloaded into `_vendor/CMTFusion/saved_models/`. Needs `kornia`.
  Trained on KAIST (vehicle-mounted RGB-thermal). Multi-scale CNN with cross-modal attention whose cost is linear in
  the number of pixels, so any input size works.

**Data** (on Drive in `DRIVE_RENDERS`, unzipped into `/content/data` on the first run):

| batch | content | used for |
|---|---|---|
| `batch_1` | 10 scenes, 1280x720 (the former `forest_batch_1`) | test videos |
| `batch_2` | scenes, 1280x1024 | test videos |
| `batch_3`, `batch_4` | 496 independent snapshots (2 views per scenario), 1280x1024 | fine-tuning; `VAL_FRACTION` of the scenarios for validation |

Training on snapshots and testing on scenes keeps the two apart completely, and the snapshots cover far more
variation (fog density and droplet size, light, camera height and angle, figure placement) than the frames of a few
scenes, whose neighbouring frames are near-duplicates. Validation holds out whole scenarios, so the two views of one
scenario are never split between training and validation.

**Thermal registration** is fitted from each annotation file's `bbox_visible` / `bbox_thermal` pairs instead of being
hard-coded: `batch_2` to `batch_4` give thermal = visible / 2, `batch_1` thermal = visible / 2 with 76 rows of
offset (its thermal view is taller than the visible one). A fit that does not explain the boxes to within a pixel
stops the notebook.

**Loss**: SwinFusion's infrared-visible fusion loss (`models/loss_vif.py`), with the **fog-free** frame in place of
the visible input. The input is always the foggy visible frame plus the thermal frame.

`loss = W_INT * |F - max(clear, ir)| + W_GRAD * |grad F - max(grad clear, grad ir)| + W_SSIM * (1 - SSIM_w)`

`SSIM_w` weights SSIM(F, clear) and SSIM(F, ir) by the mean gradient of each. The max terms ask for the brighter and
the sharper of the two sources at every pixel, so thermal hot spots are rewarded rather than traded away for
similarity to the visible frame (the failure of `rfn_nest_blender_v2`). Weights 20 / 20 / 10 as in SwinFusion.

**Outputs** (on Drive in `DRIVE_OUT` on Colab):
- `checkpoints/cmtfusion_blender_v1.pth` (fine-tuned weights) and `checkpoints/cmtfusion_blender_v1_history.csv`.
- `out/cmtfusion_blender_v1_<source>.mp4` for every test scene: `forest_<clip>` for `batch_1` (the names used so far, so
  the rows compare with earlier results), `forest_b2_<clip>` for `batch_2`.

**Scoring**: run this notebook locally with `RUN_TRAINING = False` and `RUN_FUSION = False` once, with the batches in
`fusion/data/` (folders or zips): that writes the evaluation sources `fusion/data/<source>/{rgb,ir,clear}`. Copy the
videos to `fusion/out/` and run `FUSION_EVALUATION.ipynb` per source.

**Speed:** fine-tuning and `FRAME_STEP = 1` need a GPU (Colab T4). `FRAME_STEP = 10` fuses only the frames
`FUSION_EVALUATION.ipynb` scores and repeats each one: the same scores at a tenth of the time.

| version | change |
|---|---|
| v1 | whole network (CrossFuse: fusion transformer only) fine-tuned from the published weights, SwinFusion loss with the fog-free target |


In [1]:
METHOD = 'cmtfusion_blender_v1'   # output: <METHOD>_<source>.mp4, checkpoint: <METHOD>.pth

# Data: batch_1 and batch_2 are scenes (test), batch_3 and batch_4 independent snapshots (fine-tuning)
TRAIN_BATCHES = ['batch_3', 'batch_4']
TEST_BATCHES = ['batch_1', 'batch_2']
TEST_SOURCES = None           # None = every scene of TEST_BATCHES, or a list such as ['forest_07_afternoon_steep-push']
VAL_FRACTION = 0.1            # share of the training scenarios held out to choose the best epoch
SEED = 42

# Colab: the batches are read from DRIVE_RENDERS, outputs go to DRIVE_OUT. Locally: fusion/data and fusion/out
DRIVE_RENDERS = '/content/drive/MyDrive/Vidhance Extended View/Blender renders'          # batch_1.zip ... batch_4.zip
DRIVE_OUT = '/content/drive/MyDrive/Vidhance Extended View/models_output/transformers'  # out/ and checkpoints/
LOCAL_ROOT = None             # locally: None = the repo's fusion/ folder (data/ and out/ inside it)

# Fine-tuning
RUN_TRAINING = True           # False = no training: use this METHOD's checkpoint if it exists, else the published weights
RETRAIN = False               # False = reuse a finished checkpoint of this METHOD instead of training again
EPOCHS = 20
LR = 5e-5                     # Adam; low because training starts from the published weights
BATCH_SIZE = 8
CROP = 256                    # random training crop (full resolution), flipped at random
CROPS_PER_FRAME = 4           # crops per training frame and epoch
VAL_CROP = 512                # centre crop of every validation frame
W_INT, W_GRAD, W_SSIM = 20, 20, 10   # SwinFusion's loss weights

# Output videos
RUN_FUSION = True             # False = stop after training (and after writing the evaluation sources)
SCALE = 1                     # fuse at this fraction of the input size (output is resized back to full size)
MAX_FRAMES = None             # None = all frames, or an int to fuse only the first N
FRAME_STEP = 1                # fuse every n-th frame and repeat it until the next one (keeps the frame count).
                              # 10 = exactly the frames FUSION_EVALUATION scores (EVAL_STRIDE = 10), so the scores equal
                              # a full run at a tenth of the cost; 1 = fuse every frame (a smooth video to watch)
FPS = 30.0                    # frame rate for image-sequence inputs (videos keep their own)
OVERWRITE = True              # False = skip clips whose output already exists
PREVIEW_FRAME = 100           # frame index shown in the preview at the end (a multiple of FRAME_STEP)

CMT_CKPT = None               # None -> the released weights, downloaded on first run
NORMALIZE = 'ema'             # 'frame' = repo (per-frame min..max), 'ema' = min..max smoothed over the clip
EMA = 0.8                     # weight of the running range per fused frame (only for NORMALIZE = 'ema')

In [2]:
import os, sys, json, shutil, subprocess, time, zipfile, importlib, csv, random
from concurrent.futures import ThreadPoolExecutor
from itertools import islice
from pathlib import Path

def _need(mod, pip_name=None):                       # fallback if requirements.txt was not installed
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or mod], check=True)

for _m, _p in [('numpy', None), ('cv2', 'opencv-python'), ('torch', None), ('kornia', None), ('tqdm', None), ('matplotlib', None)]:
    _need(_m, _p)

import numpy as np, cv2, torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not torch.cuda.is_available():
        raise RuntimeError('No GPU: choose Runtime > Change runtime type > T4 GPU, then run all cells again.')
    VENDOR = Path('/content/_vendor')
    RAW_DIR = Path('/content/data')                  # batch_N unzipped from DRIVE_RENDERS
    DATA_DIR = Path('/content/fusion_data')          # <source>/{rgb,ir,clear} of the test scenes, built below
    OUT_DIR = Path(DRIVE_OUT) / 'out'
    CKPT_DIR = Path(DRIVE_OUT) / 'checkpoints'
else:
    here = Path.cwd().resolve()
    fusion = next((d / 'fusion' for d in [here, *here.parents] if (d / 'fusion' / 'data').is_dir()),
                  next((d for d in [here, *here.parents] if d.name == 'fusion' and (d / 'data').is_dir()), None))
    if fusion is None:
        raise FileNotFoundError('Could not find fusion/data from %s. Start Jupyter inside the repo.' % here)
    VENDOR = fusion / 'methods' / '_vendor'          # third-party code is cloned here
    base = Path(LOCAL_ROOT) if LOCAL_ROOT else fusion
    RAW_DIR = DATA_DIR = base / 'data'
    OUT_DIR, CKPT_DIR = base / 'out', base / 'out' / 'checkpoints'

def fetch_repo(name, url):
    d = VENDOR / name
    if not d.is_dir():
        VENDOR.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1', url, str(d)], check=True)
    return d

def pick_ckpt(user_value, default):
    p = Path(user_value) if user_value else Path(default)
    if not p.exists():
        raise FileNotFoundError('Checkpoint not found: %s (set the *_CKPT option in the config cell)' % p)
    return p

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('outputs:', OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device : Tesla T4
outputs: /content/drive/MyDrive/Vidhance Extended View/models_output/transformers/out


## Data

Each batch is unzipped once (Colab: from `DRIVE_RENDERS` into `/content/data`; locally `fusion/data/<batch>/` is used,
or `fusion/data/<batch>.zip` is unzipped there, and `forest_batch_1` counts as `batch_1`). Two layouts exist:
- **scenes** (`batch_1`, `batch_2`): `<clip>/frames/{visible_haze, visible_clear, thermal_haze}/NNNN.png` and
  `<clip>/annotations.json`;
- **snapshots** (`batch_3`, `batch_4`): `{visible_fog, visible_no_fog, infrared}/<scenario>_<light>_<view>.png` and
  one `annotations.json`.

The thermal frame is registered to the visible one with `x_t = ax * x_v + bx`, `y_t = ay * y_v + by`, fitted by least
squares to the corners of every annotated figure box of that annotation file (boxes cut off by the visible frame's
edge are left out of the fit) and applied with `cv2.remap`.


In [3]:
def batch_dir(name):
    """Folder of an unzipped batch, unzipping it on first use."""
    if IN_COLAB:
        d = RAW_DIR / name
        if not (d / '.complete').exists():
            src, t0 = Path(DRIVE_RENDERS) / (name + '.zip'), time.time()
            if not src.exists():
                raise FileNotFoundError('%s not found: upload %s.zip to Drive or change DRIVE_RENDERS.' % (src, name))
            if d.exists():
                print('The copy in %s is incomplete (interrupted unzip?), replacing it.' % d)
                shutil.rmtree(d)
            RAW_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(src) as zf:
                zf.extractall(RAW_DIR)
            (d / '.complete').touch()
            print('%s unzipped in %.0f s' % (name, time.time() - t0))
        return d
    for d in (RAW_DIR / name, RAW_DIR / {'batch_1': 'forest_batch_1'}.get(name, name)):
        if d.is_dir():
            return d
    src = RAW_DIR / (name + '.zip')
    if src.exists():
        with zipfile.ZipFile(src) as zf:
            zf.extractall(RAW_DIR)
        return RAW_DIR / name
    raise FileNotFoundError('%s not found: put the %s folder or %s.zip in %s' % (name, name, name, RAW_DIR))


# ---- registration ----------------------------------------------------------------------------------------------
def _bbox_pairs(obj):
    if isinstance(obj, dict):
        if 'bbox_visible' in obj and 'bbox_thermal' in obj:
            yield obj['bbox_visible'], obj['bbox_thermal']
        for v in obj.values():
            yield from _bbox_pairs(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from _bbox_pairs(v)

def fit_registration(annotation_file, vis_w, vis_h):
    """(ax, bx, ay, by) with x_t = ax * x_v + bx and y_t = ay * y_v + by, from the annotated figure boxes."""
    pairs = list(_bbox_pairs(json.loads(Path(annotation_file).read_text())))
    if not pairs:
        raise ValueError('%s holds no bbox_visible / bbox_thermal pairs to fit the registration' % annotation_file)
    v, t = np.array([p[0] for p in pairs], float), np.array([p[1] for p in pairs], float)
    params = []
    for cols, size in (([0, 2], vis_w), ([1, 3], vis_h)):
        xv, xt = v[:, cols].ravel(), t[:, cols].ravel()
        keep = (xv > 0.5) & (xv < size - 0.5)                   # a box cut off by the visible frame is not a match
        a, b = np.polyfit(xv[keep], xt[keep], 1)
        err = np.abs(a * xv[keep] + b - xt[keep])
        if err.max() > 1.0:                                     # one refit without outliers (e.g. a clipped thermal box)
            keep2 = err <= 1.0
            a, b = np.polyfit(xv[keep][keep2], xt[keep][keep2], 1)
            err = np.abs(a * xv[keep][keep2] + b - xt[keep][keep2])
            if keep2.mean() < 0.9 or err.max() > 1.0:
                raise ValueError('%s: the boxes do not fit one linear mapping (max error %.2f px)' % (annotation_file, err.max()))
        params += [round(a, 4) + 0.0, round(b, 1) + 0.0]                    # the annotations have 0.1 px precision
    return tuple(params)

_maps = {}
def register(thermal, reg, vis_w, vis_h):
    """Thermal frame resampled onto the visible pixel grid (pixel centres at +0.5)."""
    key = (reg, vis_w, vis_h, thermal.shape)
    if key not in _maps:
        ax, bx, ay, by = reg
        mx = (ax * (np.arange(vis_w) + 0.5) + bx - 0.5).astype(np.float32)
        my = (ay * (np.arange(vis_h) + 0.5) + by - 0.5).astype(np.float32)
        _maps[key] = (np.tile(mx, (vis_h, 1)), np.tile(my[:, None], (1, vis_w)))
    return cv2.remap(thermal, *_maps[key], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def read_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError('Could not read %s' % path)
    return img


# ---- training snapshots ----------------------------------------------------------------------------------------
SNAP = ('visible_fog', 'visible_no_fog', 'infrared')

def snapshot_list(batch):
    d = batch_dir(batch)
    if not (d / SNAP[0]).is_dir():
        raise ValueError('%s has no %s/ folder: it is not a snapshot batch' % (batch, SNAP[0]))
    names = {s: sorted(p.name for p in (d / s).iterdir() if p.suffix == '.png') for s in SNAP}
    if not names[SNAP[0]] or any(names[s] != names[SNAP[0]] for s in SNAP):
        raise ValueError('%s: %s do not hold the same files' % (batch, ', '.join(SNAP)))
    w, h = cv2.imread(str(d / SNAP[0] / names[SNAP[0]][0])).shape[1::-1]
    reg = fit_registration(d / 'annotations.json', w, h)
    print('%s: %d snapshots, %dx%d, thermal x = %.4g * x + %.1f, y = %.4g * y + %.1f' % (batch, len(names[SNAP[0]]), w, h, *reg))
    return [(d, n, reg, (batch, n.split('_')[0])) for n in names[SNAP[0]]]   # group = (batch, scenario)

def load_snapshots(items):
    """(foggy visible, fog-free visible, registered thermal) as grey uint8 arrays."""
    def one(item):
        d, n, reg, _ = item
        vi = read_gray(d / 'visible_fog' / n)
        h, w = vi.shape
        return vi, read_gray(d / 'visible_no_fog' / n), register(read_gray(d / 'infrared' / n), reg, w, h)
    with ThreadPoolExecutor(4) as pool:
        return list(tqdm(pool.map(one, items), total=len(items), desc='loading snapshots', leave=False))

train_items, val_items = [], []
if RUN_TRAINING:
    items = [it for b in TRAIN_BATCHES for it in snapshot_list(b)]
    groups = sorted({it[3] for it in items})
    rng = np.random.default_rng(SEED)
    val_groups = {groups[i] for i in rng.permutation(len(groups))[:max(1, round(VAL_FRACTION * len(groups)))]}
    train_items = [it for it in items if it[3] not in val_groups]
    val_items = [it for it in items if it[3] in val_groups]
    print('train %d snapshots, val %d (%d of %d scenarios held out)' % (len(train_items), len(val_items), len(val_groups), len(groups)))


# ---- test scenes -> <source>/{rgb, ir, clear} --------------------------------------------------------------------
SCENE = ('visible_haze', 'visible_clear', 'thermal_haze')

def source_name(batch, clip):
    return 'forest_' + clip if batch == 'batch_1' else 'forest_%s_%s' % (batch.replace('batch_', 'b'), clip)

def scene_list(batch):
    d = batch_dir(batch)
    return [(source_name(batch, c.name), batch, c) for c in sorted(d.iterdir()) if (c / 'frames' / SCENE[0]).is_dir()]

def link_or_copy(src, dst):
    try:
        dst.symlink_to(src, target_is_directory=True)
    except OSError:                                         # e.g. Windows without symlink rights
        shutil.copytree(src, dst)

def export_source(batch, clip_dir, dst):
    """dst/rgb = visible_haze, dst/clear = visible_clear (linked), dst/ir = registered thermal_haze (written)."""
    frames = clip_dir / 'frames'
    names = sorted(p.name for p in (frames / SCENE[0]).iterdir() if p.suffix == '.png')
    if all((dst / s).is_dir() and len(list((dst / s).iterdir())) == len(names) for s in ('rgb', 'ir', 'clear')):
        return len(names)
    w, h = cv2.imread(str(frames / SCENE[0] / names[0])).shape[1::-1]
    reg = fit_registration(clip_dir / 'annotations.json', w, h)
    dst.mkdir(parents=True, exist_ok=True)
    for sub, stream in (('rgb', 'visible_haze'), ('clear', 'visible_clear')):
        if not (dst / sub).exists():
            link_or_copy(frames / stream, dst / sub)
    (dst / 'ir').mkdir(exist_ok=True)
    for n in tqdm(names, desc=dst.name, leave=False):
        cv2.imwrite(str(dst / 'ir' / n), register(read_gray(frames / 'thermal_haze' / n), reg, w, h))
    print('%-40s %4d frames, %dx%d, thermal x = %.4g * x + %.1f, y = %.4g * y + %.1f' % (dst.name, len(names), w, h, *reg))
    return len(names)

scenes = [s for b in TEST_BATCHES for s in scene_list(b)]
if TEST_SOURCES is not None:
    unknown = set(TEST_SOURCES) - {s[0] for s in scenes}
    if unknown:
        raise ValueError('TEST_SOURCES not in %s: %s' % (TEST_BATCHES, sorted(unknown)))
    scenes = [s for s in scenes if s[0] in TEST_SOURCES]
SOURCES = [s[0] for s in scenes]
DATA_DIR.mkdir(parents=True, exist_ok=True)
n_frames = sum(export_source(b, c, DATA_DIR / src) for src, b, c in scenes)
print('%d test sources, %d frames, in %s' % (len(SOURCES), n_frames, DATA_DIR))


batch_3: 220 snapshots, 1280x1024, thermal x = 0.5 * x + 0.0, y = 0.5 * y + 0.0
batch_4: 276 snapshots, 1280x1024, thermal x = 0.5 * x + 0.0, y = 0.5 * y + 0.0
train 432 snapshots, val 64 (16 of 156 scenarios held out)
25 test sources, 3712 frames, in /content/fusion_data


In [4]:
import urllib.request, zipfile

CMT_REPO = fetch_repo('CMTFusion', 'https://github.com/seonghyun0108/CMTFusion.git')
WEIGHTS_URL = 'https://drive.google.com/uc?export=download&id=1-yfpwxfntbwmsY8mTeYAqacowE_lL7h4'   # from the README
DEFAULT_CKPT = CMT_REPO / 'saved_models' / 'pretrained.pth'
if CMT_CKPT is None and not DEFAULT_CKPT.exists():
    DEFAULT_CKPT.parent.mkdir(parents=True, exist_ok=True)
    zip_path = DEFAULT_CKPT.parent / 'pretrained.zip'
    urllib.request.urlretrieve(WEIGHTS_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DEFAULT_CKPT.parent)
    zip_path.unlink()
def use_repo(d):
    """Puts repo d first on the import path and forgets modules imported from another vendored repo, so that
    same-named packages (DATFuse and ITFuse both have `Networks`, others `models`, `utils`) cannot be mixed up
    when several of these notebooks run in one kernel."""
    d = str(d)
    while d in sys.path:
        sys.path.remove(d)
    sys.path.insert(0, d)
    for name, mod in list(sys.modules.items()):
        locs = [getattr(mod, '__file__', None) or '']
        try:                                           # torch.classes.__path__ is not a list (torch >= 2.x)
            locs += [str(p) for p in (getattr(mod, '__path__', None) or [])]
        except TypeError:
            pass
        if any(str(VENDOR) in l for l in locs) and not any(l.startswith(d) for l in locs if l):
            del sys.modules[name]

use_repo(CMT_REPO)
print(sorted(p.name for p in CMT_REPO.iterdir()))

['.git', 'LICENSE', 'README.md', 'figs', 'losses.py', 'model.py', 'pytorch_ssim', 'saved_models', 'test.py', 'train.py', 'utils.py']


In [5]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

use_repo(CMT_REPO)                                   # in case another notebook ran in this kernel
from model import CMTFusion

cmt_ckpt = pick_ckpt(CMT_CKPT, DEFAULT_CKPT)
state = torch.load(cmt_ckpt, map_location='cpu')
state = {k[len('module.'):] if k.startswith('module.') else k: v for k, v in state.items()}   # saved from DataParallel
cmt_model = CMTFusion()
cmt_model.load_state_dict(state, strict=True)
cmt_model.eval().to(DEVICE)
print(cmt_ckpt, '| params: %.3fM' % (sum(p.numel() for p in cmt_model.parameters()) / 1e6))

device: cuda


/content/_vendor/CMTFusion/model.py:28: SyntaxWarning: "is" with 'float' literal. Did you mean "=="?
  if lef_right%2 is 0.0:
/content/_vendor/CMTFusion/model.py:37: SyntaxWarning: "is" with 'float' literal. Did you mean "=="?
  if top_bot%2 is 0.0:


/content/_vendor/CMTFusion/saved_models/pretrained.pth | params: 0.624M


In [6]:
# What is trained: the whole network, from the published weights. Its raw output spans only a narrow range and
# fuse_pair stretches every frame to 0..1, so the loss sees the output after the same per-image stretch.
TRAIN_MODULES = {'model': cmt_model}

def minmax_per_image(x):
    lo = x.amin(dim=(1, 2, 3), keepdim=True)
    hi = x.amax(dim=(1, 2, 3), keepdim=True)
    return (x - lo) / (hi - lo + 1e-8)

def forward_train(vi, ir):
    """vi, ir: (B, 1, h, w) in [0, 1] -> fused (B, 1, h, w) in [0, 1]; inputs scaled to [-1, 1] as in fuse_pair."""
    return minmax_per_image(cmt_model(vi * 2 - 1, ir * 2 - 1)[0])


## Fine-tuning

Epoch 0 is the published weights. After every epoch the validation snapshots (a centre crop of each) are scored with
the same loss, and the best epoch is saved to `<METHOD>.pth.partial`, renamed to `<METHOD>.pth` only when training has
finished, so an interrupted run never leaves a checkpoint that looks complete. BatchNorm layers keep the published
statistics (batches of a few crops would distort them). `val_ssim_clear` is the plain SSIM between the output and the
fog-free frame, for orientation only.


In [7]:
# ---- loss: SwinFusion's fusion_loss_vif with the fog-free frame as the visible image ---------------------------
_g = cv2.getGaussianKernel(11, 1.5)
_WIN = torch.tensor(_g @ _g.T, dtype=torch.float32)[None, None]
_SOBEL_X = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)[None, None]

def ssim(a, b):
    """Mean SSIM of two (B, 1, h, w) images in [0, 1]: 11x11 Gaussian window, sigma 1.5."""
    w = _WIN.to(a.device)
    mu_a, mu_b = F.conv2d(a, w), F.conv2d(b, w)
    va = F.conv2d(a * a, w) - mu_a ** 2
    vb = F.conv2d(b * b, w) - mu_b ** 2
    cov = F.conv2d(a * b, w) - mu_a * mu_b
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    return (((2 * mu_a * mu_b + C1) * (2 * cov + C2)) / ((mu_a ** 2 + mu_b ** 2 + C1) * (va + vb + C2))).mean()

def sobel(x):
    kx = _SOBEL_X.to(x.device)
    return F.conv2d(x, kx, padding=1).abs() + F.conv2d(x, kx.transpose(2, 3), padding=1).abs()

def fusion_loss(fused, clear, ir):
    g_f, g_c, g_i = sobel(fused), sobel(clear), sobel(ir)
    l_int = F.l1_loss(fused, torch.max(clear, ir))
    l_grad = F.l1_loss(g_f, torch.max(g_c, g_i))
    w_c = g_c.mean() / (g_c.mean() + g_i.mean() + 1e-8)
    l_ssim = 1 - (w_c * ssim(clear, fused) + (1 - w_c) * ssim(ir, fused))
    return W_INT * l_int + W_GRAD * l_grad + W_SSIM * l_ssim, (l_int.item(), l_grad.item(), l_ssim.item())


# ---- training --------------------------------------------------------------------------------------------------
def set_mode(train):
    for m in TRAIN_MODULES.values():
        m.train(train)
        for sub in m.modules():
            if isinstance(sub, nn.modules.batchnorm._BatchNorm):
                sub.eval()                              # keep the published BatchNorm statistics

def to_t(arrays):
    return torch.from_numpy(np.stack(arrays)).float().div(255)[:, None].to(DEVICE)

def train_batches(data):
    order = np.random.permutation(np.repeat(np.arange(len(data)), CROPS_PER_FRAME))
    for start in range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE):
        crops = ([], [], [])
        for idx in order[start:start + BATCH_SIZE]:
            h, w = data[idx][0].shape
            c = min(CROP, h, w)
            y, x = np.random.randint(h - c + 1), np.random.randint(w - c + 1)
            flip = np.random.random() < 0.5
            for out, img in zip(crops, data[idx]):
                crop = img[y:y + c, x:x + c]
                out.append(crop[:, ::-1].copy() if flip else crop)
        yield tuple(to_t(c) for c in crops)

@torch.no_grad()
def validate(data):
    set_mode(False)
    total, ssim_clear = 0.0, 0.0
    for vi, clear, ir in data:
        h, w = vi.shape
        c = min(VAL_CROP, h, w)
        y, x = (h - c) // 2, (w - c) // 2
        vi_t, clear_t, ir_t = (to_t([a[y:y + c, x:x + c]]) for a in (vi, clear, ir))
        fused = forward_train(vi_t, ir_t)
        total += fusion_loss(fused, clear_t, ir_t)[0].item()
        ssim_clear += ssim(clear_t, fused.clamp(0, 1)).item()
    return total / len(data), ssim_clear / len(data)

def save_weights(path):
    torch.save({k: m.state_dict() for k, m in TRAIN_MODULES.items()}, path)

def load_weights(path):
    state = torch.load(path, map_location='cpu')
    for k, m in TRAIN_MODULES.items():
        m.load_state_dict(state[k])

CKPT_DIR.mkdir(parents=True, exist_ok=True)
FT_CKPT = CKPT_DIR / (METHOD + '.pth')
if FT_CKPT.exists() and not RETRAIN:
    load_weights(FT_CKPT)
    print('Loaded', FT_CKPT, '(set RETRAIN = True to train again)')
elif not RUN_TRAINING:
    print('RUN_TRAINING = False and no %s: the published weights are used.' % FT_CKPT.name)
else:
    train_data, val_data = load_snapshots(train_items), load_snapshots(val_items)
    params = [p for m in TRAIN_MODULES.values() for p in m.parameters()]
    for p in params:
        p.requires_grad_(True)
    optimizer = torch.optim.Adam(params, lr=LR)
    partial = FT_CKPT.with_name(FT_CKPT.name + '.partial')
    best_val, val_ssim = validate(val_data)
    best_epoch = 0
    save_weights(partial)
    history = [dict(epoch=0, loss=None, l_int=None, l_grad=None, l_ssim=None, val_loss=best_val, val_ssim_clear=val_ssim)]
    print('training %d parameters on %d snapshots' % (sum(p.numel() for p in params), len(train_data)))
    print('epoch  0  published weights                                   val %.4f  val SSIM to fog-free %.3f' % (best_val, val_ssim))
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        set_mode(True)
        sums, steps = np.zeros(4), 0
        for vi, clear, ir in tqdm(train_batches(train_data), leave=False, desc='epoch %d' % epoch,
                                  total=len(train_data) * CROPS_PER_FRAME // BATCH_SIZE):
            loss, parts = fusion_loss(forward_train(vi, ir), clear, ir)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            sums += (loss.item(), *parts)
            steps += 1
        val, val_ssim = validate(val_data)
        mark = ''
        if val < best_val:
            best_val, best_epoch, mark = val, epoch, '  saved'
            save_weights(partial)
        m = sums / max(steps, 1)
        history.append(dict(epoch=epoch, loss=m[0], l_int=m[1], l_grad=m[2], l_ssim=m[3], val_loss=val, val_ssim_clear=val_ssim))
        print('epoch %2d  loss %.4f (int %.4f, grad %.4f, 1-ssim %.4f)  val %.4f  val SSIM to fog-free %.3f  (%.0f s)%s'
              % (epoch, *m, val, val_ssim, time.time() - t0, mark))
    os.replace(partial, FT_CKPT)                       # only now does the checkpoint count as finished
    with open(CKPT_DIR / (METHOD + '_history.csv'), 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    load_weights(FT_CKPT)
    print('best epoch %d: val %.4f (published weights: %.4f)' % (best_epoch, best_val, history[0]['val_loss'])
          if best_epoch else 'WARNING: no epoch beat the published weights, so %s holds the published weights.' % METHOD)
    del train_data, val_data, optimizer
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
set_mode(False)


loading snapshots:   0%|          | 0/432 [00:00<?, ?it/s]

loading snapshots:   0%|          | 0/64 [00:00<?, ?it/s]

training 623989 parameters on 432 snapshots
epoch  0  published weights                                   val 9.9138  val SSIM to fog-free 0.248


epoch 1:   0%|          | 0/216 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Process 511 has 2.49 GiB memory in use. Process 18367 has 9.03 GiB memory in use. Including non-PyTorch memory, this process has 3.04 GiB memory in use. Of the allocated memory 2.85 GiB is allocated by PyTorch, and 63.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
@torch.no_grad()
def fuse_pair(vis_gray, ir_gray):
    # test.py / utils.get_test_images: grayscale, ToTensor, Normalize(0.5, 0.5) -> [-1, 1]; model(visible, infrared);
    # the first of the three outputs is the full-resolution fusion
    to_t = lambda a: (torch.from_numpy(a.astype(np.float32) / 255.0) * 2 - 1)[None, None].to(DEVICE)
    out = cmt_model(to_t(vis_gray), to_t(ir_gray))[0][0, 0].float().cpu()
    return np.clip(np.round(stretch(out).numpy() * 255.0), 0, 255).astype(np.uint8)

# The repo stretches every output frame to its own min..max. On video that can pump brightness when the extreme pixels
# change, so NORMALIZE = 'ema' smooths min and max over the clip instead ('frame' = exactly the repo's behaviour).
_range = None

def reset_state():
    global _range
    _range = None

def stretch(o):
    global _range
    lo, hi = float(o.min()), float(o.max())
    if NORMALIZE == 'ema':
        _range = (lo, hi) if _range is None else (EMA * _range[0] + (1 - EMA) * lo, EMA * _range[1] + (1 - EMA) * hi)
        lo, hi = _range
    return (o - lo) / (hi - lo + 1e-8)

In [ ]:
# Reading a scene from fusion/data/<SOURCE>/
import re
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
RGB_DIRS = ('rgb', 'visible', 'vis', 'vi', 'color')
IR_DIRS = ('ir', 'thermal', 'infrared', 't')
RGB_VIDEOS = ('visible.mp4', 'rgb.mp4')
IR_VIDEOS = ('thermal.mp4', 'ir.mp4')

def rescale(img):
    return img if SCALE == 1 else cv2.resize(img, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)

def _natural(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', p.name)]

def _find(d, names, want_dir):
    for p in sorted(d.iterdir()):
        if p.name.lower() in names and (p.is_dir() if want_dir else p.is_file()):
            return p

def image_pairs(rgb_dir, ir_dir):
    rgb = {p.name: p for p in sorted(rgb_dir.iterdir(), key=_natural) if p.suffix.lower() in IMG_EXT}
    ir = {p.name: p for p in sorted(ir_dir.iterdir(), key=_natural) if p.suffix.lower() in IMG_EXT}
    names = sorted(set(rgb) & set(ir), key=lambda n: _natural(Path(n)))
    if names:
        pairs = [(rgb[n], ir[n]) for n in names]
    else:
        print('WARNING: rgb/ and ir/ share no filenames, pairing by sorted position')
        a, b = sorted(rgb.values(), key=_natural), sorted(ir.values(), key=_natural)
        pairs = list(zip(a, b))
    def gen():
        for a, b in pairs:
            v, t = cv2.imread(str(a), cv2.IMREAD_COLOR), cv2.imread(str(b), cv2.IMREAD_GRAYSCALE)
            if v is None or t is None:
                raise IOError('Could not read %s / %s' % (a, b))
            yield v, t
    return len(pairs), FPS, gen()

def video_pairs(rgb_path, ir_path):
    a, b = cv2.VideoCapture(str(rgb_path)), cv2.VideoCapture(str(ir_path))
    if not (a.isOpened() and b.isOpened()):
        raise IOError('Could not open %s / %s' % (rgb_path, ir_path))
    n = min(int(a.get(cv2.CAP_PROP_FRAME_COUNT)), int(b.get(cv2.CAP_PROP_FRAME_COUNT)))
    fps = a.get(cv2.CAP_PROP_FPS) or FPS
    def gen():
        try:
            while True:
                ok1, v = a.read(); ok2, t = b.read()
                if not (ok1 and ok2):
                    break
                yield v, cv2.cvtColor(t, cv2.COLOR_BGR2GRAY)
        finally:
            a.release(); b.release()
    return n, fps, gen()

def open_pairs(src_dir):
    rgb_dir, ir_dir = _find(src_dir, RGB_DIRS, True), _find(src_dir, IR_DIRS, True)
    if rgb_dir and ir_dir:
        return image_pairs(rgb_dir, ir_dir)
    rgb_v, ir_v = _find(src_dir, RGB_VIDEOS, False), _find(src_dir, IR_VIDEOS, False)
    if rgb_v and ir_v:
        return video_pairs(rgb_v, ir_v)
    raise FileNotFoundError('%s needs rgb/ + ir/ folders or visible.mp4 + thermal.mp4, found: %s'
                            % (src_dir, sorted(p.name for p in src_dir.iterdir())))

def make_writer(path, fps, size):
    for code in ('avc1', 'mp4v'):                   # H.264 if this OpenCV build has it, else MPEG-4 (an 'avc1 / Failed to initialize VideoWriter' log line is harmless)
        w = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*code), fps, size)
        if w.isOpened():
            return w
    raise RuntimeError('OpenCV could not open a video writer for %s' % path)

In [ ]:
if not RUN_FUSION:
    SOURCES = []                                     # nothing to fuse
OUT_DIR.mkdir(parents=True, exist_ok=True)
gray = lambda bgr: cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
previews = {}
for SOURCE in SOURCES:
    src_dir = DATA_DIR / SOURCE
    if not src_dir.is_dir():
        raise FileNotFoundError('fusion/data/%s not found. Available: %s'
                                % (SOURCE, sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())))
    out_path = OUT_DIR / ('%s_%s.mp4' % (METHOD, SOURCE))
    if out_path.exists() and not OVERWRITE:
        print('exists, skipping:', out_path)
        continue
    n_total, fps, frames = open_pairs(src_dir)
    if MAX_FRAMES:
        n_total = min(n_total, MAX_FRAMES) if n_total else MAX_FRAMES
    reset_state()                                      # per-clip state of the output normalization, if any
    tmp_path = OUT_DIR / ('.%s_%s.partial.mp4' % (METHOD, SOURCE))   # renamed only when finished
    writer, done, fused, g, t0 = None, 0, 0, None, time.time()
    try:
        for i, (rgb, ir) in enumerate(tqdm(islice(frames, MAX_FRAMES), total=n_total or None, desc=SOURCE)):
            H, W = rgb.shape[:2]
            if i % FRAME_STEP == 0:
                if ir.shape[:2] != (H, W):
                    ir = cv2.resize(ir, (W, H))
                g = fuse_pair(rescale(gray(rgb)), rescale(ir))
                if DEVICE == 'cuda':
                    torch.cuda.synchronize()
                if g.shape[:2] != (H, W):
                    g = cv2.resize(g, (W, H), interpolation=cv2.INTER_LINEAR)
                fused += 1
            if writer is None:
                writer = make_writer(tmp_path, fps, (W, H))
            writer.write(cv2.cvtColor(g, cv2.COLOR_GRAY2BGR))
            if i == PREVIEW_FRAME or (SOURCE not in previews and i == (n_total or 1) - 1):
                previews[SOURCE] = (rgb, ir, g, i)
            done += 1
    finally:
        if writer is not None:
            writer.release()
        frames.close()
    if done == 0:
        raise RuntimeError('No frames were read from %s' % src_dir)
    os.replace(tmp_path, out_path)
    sec = time.time() - t0
    print('%s: %d frames written, %d fused in %.1fs (%.2f s per fused frame on %s) -> %s'
          % (SOURCE, done, fused, sec, sec / fused, DEVICE, out_path.name))

In [ ]:
for SOURCE, (rgb, ir, g, i) in previews.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, img, title in zip(axes, [cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB), ir, g], ['visible', 'thermal', 'fused: ' + METHOD]):
        ax.imshow(img, cmap=None if img.ndim == 3 else 'gray', vmin=0, vmax=255); ax.set_title('%s (%s, frame %d)' % (title, SOURCE, i)); ax.axis('off')
    plt.show()
print('Next: run fusion/FUSION_EVALUATION.ipynb (once per SOURCE) to score', METHOD)